
# Step 02 — Compositional Analysis of Semantic Attention

The semantic gaze vector is

\[
G=[g_H,g_P,g_C,g_O], \qquad
g_H+g_P+g_C+g_O=1.
\]

Because these four components are constrained to sum to one, they are **compositional data**.  
This notebook analyzes them in log-ratio geometry rather than treating the four proportions as independent variables.

## Main interpretable ILR balances

We use three orthonormal balances:

### Balance 1 — Semantic AOIs vs Other/background

\[
B_1=
\sqrt{\frac{3}{4}}
\log
\left(
\frac{(g_Hg_Pg_C)^{1/3}}{g_O}
\right)
\]

Higher \(B_1\) means relatively more attention to **Headline + Product + CTA** than to background/Other.

### Balance 2 — CTA vs Headline + Product

\[
B_2=
\sqrt{\frac{2}{3}}
\log
\left(
\frac{g_C}{(g_Hg_P)^{1/2}}
\right)
\]

Higher \(B_2\) means relatively stronger CTA attention.

### Balance 3 — Headline vs Product

\[
B_3=
\sqrt{\frac{1}{2}}
\log
\left(
\frac{g_H}{g_P}
\right)
\]

Higher \(B_3\) means relatively stronger Headline than Product attention.

The notebook also performs:

- zero diagnostics;
- multiplicative zero replacement;
- sensitivity analysis to the zero-replacement magnitude;
- mixed-effects models;
- all A–E pairwise contrasts with BH/FDR;
- Aitchison-distance summaries;
- publication-ready visualizations.


In [ ]:

from pathlib import Path
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 300)

print("Working directory:", Path.cwd().resolve())


## 1. Locate and load the balanced QC-pass dataset

In [ ]:

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for base in candidates:
        if (base / "semantic_attention_analysis_outputs").is_dir():
            return base

    for base in candidates[:4]:
        try:
            for child in base.iterdir():
                if child.is_dir() and (child / "semantic_attention_analysis_outputs").is_dir():
                    return child
        except Exception:
            pass

    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate semantic_attention_analysis_outputs. "
        "Set PROJECT_ROOT manually if needed."
    )

DATA_DIR = PROJECT_ROOT / "semantic_attention_analysis_outputs"

candidate_files = [
    DATA_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES_QC_PASS.csv",
    DATA_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES.csv",
]

DATA_FILE = next((p for p in candidate_files if p.exists()), None)

if DATA_FILE is None:
    raise FileNotFoundError(
        "Balanced master dataset not found. Expected one of:\n"
        + "\n".join(str(p) for p in candidate_files)
    )

OUTPUT_DIR = Path.cwd().resolve() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_FILE)

required = ["participant_id", "condition", "image_id", "g_H", "g_P", "g_C", "g_O"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for c in ["g_H", "g_P", "g_C", "g_O"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "row_qc_pass" in df.columns:
    df = df[df["row_qc_pass"].astype(bool)].copy()

df["condition"] = pd.Categorical(
    df["condition"],
    categories=["A", "B", "C", "D", "E"],
    ordered=True
)

print("DATA_FILE:", DATA_FILE)
print("Rows:", len(df))
print("Images:", df["image_id"].nunique())
print("Unique participants:", df["participant_id"].nunique())
print("Conditions:", list(df["condition"].dropna().unique()))


## 2. Zero diagnostics

In [ ]:

PARTS = ["g_H", "g_P", "g_C", "g_O"]

zero_counts = pd.DataFrame({
    "component": PARTS,
    "n_zero": [(df[c] == 0).sum() for c in PARTS],
    "pct_zero": [100 * (df[c] == 0).mean() for c in PARTS],
    "min_positive": [
        df.loc[df[c] > 0, c].min() if (df[c] > 0).any() else np.nan
        for c in PARTS
    ],
})

display(zero_counts)
zero_counts.to_csv(OUTPUT_DIR / "zero_diagnostics.csv", index=False)

print("Rows containing at least one zero:",
      int((df[PARTS] == 0).any(axis=1).sum()),
      "of", len(df))



## 3. Multiplicative zero replacement

Observed gaze compositions can contain exact zeros because a participant did not fixate a semantic AOI during the exposure.

Log-ratio transforms require strictly positive components. We therefore use **multiplicative replacement**:

- zeros receive a small value \(\delta\);
- the positive components are rescaled proportionally;
- each row remains closed to 1.

The main analysis uses:

\[
\delta = 0.5 \times \text{smallest positive observed proportion}.
\]

Later, the notebook repeats the main summaries using smaller replacement magnitudes to check sensitivity.


In [ ]:

positive_values = df[PARTS].to_numpy()
positive_values = positive_values[positive_values > 0]

if len(positive_values) == 0:
    raise RuntimeError("No positive compositional values found.")

GLOBAL_MIN_POSITIVE = float(np.min(positive_values))
MAIN_DELTA = 0.5 * GLOBAL_MIN_POSITIVE

print("Smallest positive observed proportion:", GLOBAL_MIN_POSITIVE)
print("Main zero-replacement delta:", MAIN_DELTA)

def multiplicative_zero_replacement(X, delta):
    """
    Row-wise multiplicative replacement.

    For a row with z zeros:
      zero parts -> delta
      positive parts -> positive * (1 - z*delta) / sum(positive)

    Then close numerically to exactly 1.
    """
    X = np.asarray(X, dtype=float).copy()
    out = np.empty_like(X)

    for i, row in enumerate(X):
        if np.any(row < 0):
            raise ValueError(f"Negative composition found in row {i}")

        zero_mask = (row == 0)
        z = int(zero_mask.sum())

        if z == 0:
            s = row.sum()
            if s <= 0:
                raise ValueError(f"Invalid all-zero row {i}")
            out[i] = row / s
            continue

        if z * delta >= 1:
            raise ValueError(
                f"delta={delta} too large for row {i} with {z} zeros."
            )

        positive_mask = ~zero_mask
        positive_sum = row[positive_mask].sum()

        if positive_sum <= 0:
            raise ValueError(f"Row {i} contains no positive component.")

        new_row = np.zeros_like(row)
        new_row[zero_mask] = delta
        new_row[positive_mask] = (
            row[positive_mask] * (1 - z * delta) / positive_sum
        )

        new_row = new_row / new_row.sum()
        out[i] = new_row

    return out

X_raw = df[PARTS].to_numpy(dtype=float)
X = multiplicative_zero_replacement(X_raw, MAIN_DELTA)

print("Minimum transformed component:", X.min())
print("Maximum row-sum error:", np.abs(X.sum(axis=1) - 1).max())

for j, c in enumerate(PARTS):
    df[c + "_zr"] = X[:, j]


## 4. Compute CLR and interpretable ILR balances

In [ ]:

def clr_transform(X):
    logx = np.log(X)
    return logx - logx.mean(axis=1, keepdims=True)

def interpretable_ilr(X):
    H, P, C, O = X[:,0], X[:,1], X[:,2], X[:,3]

    b1 = np.sqrt(3/4) * np.log(((H * P * C) ** (1/3)) / O)
    b2 = np.sqrt(2/3) * np.log(C / np.sqrt(H * P))
    b3 = np.sqrt(1/2) * np.log(H / P)

    return np.column_stack([b1, b2, b3])

CLR = clr_transform(X)
ILR = interpretable_ilr(X)

for j, name in enumerate(["clr_H", "clr_P", "clr_C", "clr_O"]):
    df[name] = CLR[:, j]

df["B1_semantic_vs_other"] = ILR[:, 0]
df["B2_CTA_vs_HP"] = ILR[:, 1]
df["B3_H_vs_P"] = ILR[:, 2]

BALANCES = [
    "B1_semantic_vs_other",
    "B2_CTA_vs_HP",
    "B3_H_vs_P",
]

BALANCE_LABELS = {
    "B1_semantic_vs_other": "B1: Semantic AOIs vs Other",
    "B2_CTA_vs_HP": "B2: CTA vs Headline+Product",
    "B3_H_vs_P": "B3: Headline vs Product",
}

display(
    df[
        ["condition", "image_id", "participant_id",
         *PARTS, *BALANCES]
    ].head()
)

df.to_csv(
    OUTPUT_DIR / "compositional_dataset_with_ILR.csv",
    index=False
)


## 5. Descriptive ILR statistics

In [ ]:

balance_summary = (
    df.groupby("condition", observed=True)[BALANCES]
      .agg(["count", "mean", "std", "median", "min", "max"])
)

display(balance_summary)
balance_summary.to_csv(
    OUTPUT_DIR / "ILR_descriptive_statistics_by_condition.csv"
)


## 6. Visualization — ILR balance distributions

In [ ]:

rng = np.random.default_rng(42)
conditions = ["A", "B", "C", "D", "E"]

for balance in BALANCES:
    groups = [
        df.loc[df["condition"] == cond, balance].dropna().to_numpy()
        for cond in conditions
    ]

    fig, ax = plt.subplots(figsize=(10, 6))
    positions = np.arange(1, 6)

    ax.violinplot(
        groups,
        positions=positions,
        widths=0.8,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    ax.boxplot(
        groups,
        positions=positions,
        widths=0.18,
        showfliers=False,
    )

    for pos, vals in zip(positions, groups):
        jitter = rng.normal(0, 0.045, size=len(vals))
        ax.scatter(
            np.full(len(vals), pos) + jitter,
            vals,
            s=8,
            alpha=0.16,
        )

    ax.axhline(0, linewidth=1)
    ax.set_xticks(positions)
    ax.set_xticklabels(conditions)
    ax.set_xlabel("Condition")
    ax.set_ylabel("ILR balance")
    ax.set_title(BALANCE_LABELS[balance])
    ax.grid(axis="y", alpha=0.2)

    fig.tight_layout()
    stem = f"ILR_distribution_{balance}"
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()



## 7. Mixed-effects models on ILR balances

For each balance:

\[
B_{p,i}
=
\beta_0+
\beta_{\text{Condition}}
+
u_p+
v_i+
\epsilon
\]

where:

- \(u_p\) is the participant random intercept;
- \(v_i\) is the advertisement/image variance component.

Condition A is the treatment reference.


In [ ]:

mixed_models = {}
coef_tables = []

for balance in BALANCES:
    print("\n" + "=" * 100)
    print(BALANCE_LABELS[balance])
    print("=" * 100)

    formula = f"{balance} ~ C(condition, Treatment(reference='A'))"

    try:
        model = smf.mixedlm(
            formula,
            data=df,
            groups=df["participant_id"],
            vc_formula={"image": "0 + C(image_id)"},
            re_formula="1",
        )

        result = model.fit(
            reml=False,
            method="lbfgs",
            maxiter=2500,
            disp=False,
        )

        mixed_models[balance] = result
        print(result.summary())

        ci = result.conf_int()
        table = pd.DataFrame({
            "balance": balance,
            "term": result.params.index,
            "estimate": result.params.values,
            "ci95_low": ci.iloc[:, 0].values,
            "ci95_high": ci.iloc[:, 1].values,
            "p_value": result.pvalues.values,
        })
        coef_tables.append(table)

    except Exception as e:
        print("MODEL FAILED:", type(e).__name__, str(e))

if coef_tables:
    mixed_coef = pd.concat(coef_tables, ignore_index=True)
    display(mixed_coef)
    mixed_coef.to_csv(
        OUTPUT_DIR / "ILR_mixed_effects_coefficients.csv",
        index=False
    )


## 8. All A–E pairwise mixed-model contrasts + BH/FDR

In [ ]:

def fixed_effect_pairwise_contrast(result, c1, c2):
    """
    Estimate condition mean contrast c1 - c2 from a Treatment(A) MixedLM.

    Uses only fixed-effect coefficients/covariance.
    """
    fe_names = list(result.fe_params.index)
    beta = result.fe_params.to_numpy()
    cov = result.cov_params().loc[fe_names, fe_names].to_numpy()

    def design(cond):
        x = np.zeros(len(fe_names))
        if "Intercept" in fe_names:
            x[fe_names.index("Intercept")] = 1.0

        if cond != "A":
            term = f"C(condition, Treatment(reference='A'))[T.{cond}]"
            if term not in fe_names:
                raise KeyError(f"Missing fixed-effect term: {term}")
            x[fe_names.index(term)] = 1.0
        return x

    L = design(c1) - design(c2)
    est = float(L @ beta)
    var = float(L @ cov @ L)
    se = np.sqrt(max(var, 0))

    if se > 0:
        z = est / se
        p = 2 * stats.norm.sf(abs(z))
        lo = est - 1.96 * se
        hi = est + 1.96 * se
    else:
        z = np.nan
        p = np.nan
        lo = np.nan
        hi = np.nan

    return est, se, z, p, lo, hi

contrast_rows = []

for balance, result in mixed_models.items():
    for c1, c2 in itertools.combinations(conditions, 2):
        est, se, z, p, lo, hi = fixed_effect_pairwise_contrast(
            result, c1, c2
        )

        contrast_rows.append({
            "balance": balance,
            "condition_1": c1,
            "condition_2": c2,
            "contrast": f"{c1} - {c2}",
            "estimate": est,
            "SE": se,
            "z": z,
            "p_raw": p,
            "ci95_low": lo,
            "ci95_high": hi,
        })

contrasts = pd.DataFrame(contrast_rows)
contrasts["q_BH"] = np.nan
contrasts["reject_FDR_0_05"] = False

# Correct the 10 condition contrasts separately within each ILR balance.
for balance in BALANCES:
    mask = contrasts["balance"] == balance
    valid = mask & contrasts["p_raw"].notna()

    if valid.any():
        reject, qvals, _, _ = multipletests(
            contrasts.loc[valid, "p_raw"],
            alpha=0.05,
            method="fdr_bh",
        )
        contrasts.loc[valid, "q_BH"] = qvals
        contrasts.loc[valid, "reject_FDR_0_05"] = reject

display(
    contrasts.sort_values(["balance", "q_BH", "p_raw"])
)

contrasts.to_csv(
    OUTPUT_DIR / "ILR_pairwise_mixed_model_contrasts_BH_FDR.csv",
    index=False
)


## 9. Forest plots of Condition C contrasts

In [ ]:

for balance in BALANCES:
    sub = contrasts[
        (contrasts["balance"] == balance)
        & (
            (contrasts["condition_1"] == "C")
            | (contrasts["condition_2"] == "C")
        )
    ].copy()

    # Convert every row to C - comparator for consistent interpretation.
    rows = []
    for _, r in sub.iterrows():
        if r["condition_1"] == "C":
            comparator = r["condition_2"]
            est = r["estimate"]
            lo = r["ci95_low"]
            hi = r["ci95_high"]
        else:
            comparator = r["condition_1"]
            est = -r["estimate"]
            lo = -r["ci95_high"]
            hi = -r["ci95_low"]

        rows.append({
            "comparator": comparator,
            "estimate": est,
            "ci95_low": lo,
            "ci95_high": hi,
            "q_BH": r["q_BH"],
        })

    plot_df = pd.DataFrame(rows).sort_values("comparator")
    y = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(9, 5))
    effects = plot_df["estimate"].to_numpy()
    lo = plot_df["ci95_low"].to_numpy()
    hi = plot_df["ci95_high"].to_numpy()
    xerr = np.vstack([effects-lo, hi-effects])

    ax.errorbar(
        effects, y, xerr=xerr,
        fmt="o", capsize=4, linewidth=1.5
    )
    ax.axvline(0, linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(
        [f"C − {x}" for x in plot_df["comparator"]]
    )
    ax.set_xlabel("Mixed-model ILR contrast")
    ax.set_title(BALANCE_LABELS[balance])
    ax.grid(axis="x", alpha=0.2)

    for yi, (_, r) in enumerate(plot_df.iterrows()):
        anchor = r["ci95_high"] if np.isfinite(r["ci95_high"]) else r["estimate"]
        qtxt = "NA" if pd.isna(r["q_BH"]) else f"{r['q_BH']:.3g}"
        ax.text(anchor, yi, f"   q={qtxt}", va="center", fontsize=8)

    fig.tight_layout()
    stem = f"ILR_forest_C_vs_others_{balance}"
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()



## 10. Aitchison distance

For positive closed compositions \(x,y\), Aitchison distance is Euclidean distance in CLR space:

\[
d_A(x,y)=\|\operatorname{clr}(x)-\operatorname{clr}(y)\|_2.
\]

We compute:

1. the centroid composition of each condition;
2. the Aitchison distance between condition centroids;
3. within-condition dispersion around each centroid.


In [ ]:

def closure(v):
    v = np.asarray(v, dtype=float)
    return v / v.sum()

def geometric_center(X):
    """
    Compositional center: closure of component-wise geometric means.
    """
    gm = np.exp(np.mean(np.log(X), axis=0))
    return closure(gm)

def clr_one(v):
    lv = np.log(v)
    return lv - lv.mean()

centers = {}
center_rows = []

for cond in conditions:
    Xc = df.loc[
        df["condition"] == cond,
        [c + "_zr" for c in PARTS]
    ].to_numpy()

    center = geometric_center(Xc)
    centers[cond] = center

    center_rows.append({
        "condition": cond,
        "center_H": center[0],
        "center_P": center[1],
        "center_C": center[2],
        "center_O": center[3],
    })

center_df = pd.DataFrame(center_rows)
display(center_df)

distance_matrix = pd.DataFrame(
    index=conditions,
    columns=conditions,
    dtype=float
)

for c1 in conditions:
    for c2 in conditions:
        distance_matrix.loc[c1, c2] = np.linalg.norm(
            clr_one(centers[c1]) - clr_one(centers[c2])
        )

print("Aitchison distance between condition compositional centers:")
display(distance_matrix)

dispersion_rows = []

for cond in conditions:
    Xc = df.loc[
        df["condition"] == cond,
        [c + "_zr" for c in PARTS]
    ].to_numpy()

    center_clr = clr_one(centers[cond])
    clr_c = clr_transform(Xc)
    dists = np.linalg.norm(clr_c - center_clr, axis=1)

    dispersion_rows.append({
        "condition": cond,
        "n": len(dists),
        "mean_aitchison_to_center": dists.mean(),
        "sd_aitchison_to_center": dists.std(ddof=1),
        "median_aitchison_to_center": np.median(dists),
    })

dispersion = pd.DataFrame(dispersion_rows)
display(dispersion)

center_df.to_csv(
    OUTPUT_DIR / "condition_compositional_centers.csv",
    index=False
)
distance_matrix.to_csv(
    OUTPUT_DIR / "condition_centroid_aitchison_distances.csv"
)
dispersion.to_csv(
    OUTPUT_DIR / "within_condition_aitchison_dispersion.csv",
    index=False
)


## 11. Visualization — Aitchison-distance heatmap

In [ ]:

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(distance_matrix.to_numpy(), aspect="equal")

ax.set_xticks(np.arange(len(conditions)))
ax.set_xticklabels(conditions)
ax.set_yticks(np.arange(len(conditions)))
ax.set_yticklabels(conditions)

ax.set_xlabel("Condition")
ax.set_ylabel("Condition")
ax.set_title("Aitchison Distance Between Condition Centers")

for i in range(len(conditions)):
    for j in range(len(conditions)):
        ax.text(
            j, i,
            f"{distance_matrix.iloc[i,j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

fig.colorbar(im, ax=ax, label="Aitchison distance")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "aitchison_distance_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
fig.savefig(
    OUTPUT_DIR / "aitchison_distance_heatmap.pdf",
    bbox_inches="tight"
)
plt.show()


## 12. Image-level ILR sensitivity analysis

In [ ]:

image_balance_means = (
    df.groupby(["image_id", "condition"], observed=True)[BALANCES]
      .mean()
      .reset_index()
)

image_balance_means.to_csv(
    OUTPUT_DIR / "image_condition_ILR_means.csv",
    index=False
)

for balance in BALANCES:
    wide = (
        image_balance_means
        .pivot(index="image_id", columns="condition", values=balance)
        .reindex(columns=conditions)
        .dropna(how="any")
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(conditions))

    for _, row in wide.iterrows():
        ax.plot(
            x,
            row.to_numpy(),
            marker="o",
            linewidth=0.9,
            alpha=0.28
        )

    overall = wide.mean(axis=0).to_numpy()
    ax.plot(
        x,
        overall,
        marker="o",
        linewidth=3,
        label="Overall image-level mean"
    )

    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(conditions)
    ax.set_xlabel("Condition")
    ax.set_ylabel("Mean ILR balance")
    ax.set_title(
        f"Advertisement-level trajectories — {BALANCE_LABELS[balance]}"
    )
    ax.grid(alpha=0.2)
    ax.legend()

    fig.tight_layout()
    stem = f"image_trajectories_{balance}"
    fig.savefig(
        OUTPUT_DIR / f"{stem}.png",
        dpi=300,
        bbox_inches="tight"
    )
    fig.savefig(
        OUTPUT_DIR / f"{stem}.pdf",
        bbox_inches="tight"
    )
    plt.show()



## 13. Zero-replacement sensitivity analysis

The formal conclusions should not depend heavily on one arbitrary zero-replacement value.

We repeat the condition-level ILR means using:

\[
\delta \in
\{0.5,\,0.25,\,0.10\}
\times
\text{minimum positive observed proportion}.
\]

Large changes across these settings would signal sensitivity to zero handling.


In [ ]:

delta_factors = [0.50, 0.25, 0.10]
sens_rows = []

for factor in delta_factors:
    delta = factor * GLOBAL_MIN_POSITIVE
    Xs = multiplicative_zero_replacement(X_raw, delta)
    Zs = interpretable_ilr(Xs)

    temp = df[["condition"]].copy()
    temp["B1"] = Zs[:,0]
    temp["B2"] = Zs[:,1]
    temp["B3"] = Zs[:,2]

    means = (
        temp.groupby("condition", observed=True)[["B1","B2","B3"]]
        .mean()
    )

    for cond in conditions:
        sens_rows.append({
            "delta_factor": factor,
            "delta": delta,
            "condition": cond,
            "B1_mean": means.loc[cond, "B1"],
            "B2_mean": means.loc[cond, "B2"],
            "B3_mean": means.loc[cond, "B3"],
        })

sensitivity = pd.DataFrame(sens_rows)
display(sensitivity)

sensitivity.to_csv(
    OUTPUT_DIR / "zero_replacement_sensitivity.csv",
    index=False
)

for balance_short, title in [
    ("B1_mean", "B1: Semantic AOIs vs Other"),
    ("B2_mean", "B2: CTA vs Headline+Product"),
    ("B3_mean", "B3: Headline vs Product"),
]:
    fig, ax = plt.subplots(figsize=(9, 5))

    for cond in conditions:
        sub = sensitivity[
            sensitivity["condition"] == cond
        ].sort_values("delta_factor")

        ax.plot(
            sub["delta_factor"],
            sub[balance_short],
            marker="o",
            label=cond
        )

    ax.set_xlabel("Zero-replacement factor × minimum positive proportion")
    ax.set_ylabel("Condition mean ILR balance")
    ax.set_title(f"Zero-Replacement Sensitivity — {title}")
    ax.grid(alpha=0.2)
    ax.legend(title="Condition")

    fig.tight_layout()
    stem = f"zero_sensitivity_{balance_short}"
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()


## 14. Compact interpretation table

In [ ]:

interpretation = pd.DataFrame([
    {
        "balance": "B1_semantic_vs_other",
        "positive_direction": "Relatively more gaze to Headline+Product+CTA than Other/background",
        "negative_direction": "Relatively more gaze to Other/background",
    },
    {
        "balance": "B2_CTA_vs_HP",
        "positive_direction": "Relatively more CTA gaze than Headline+Product",
        "negative_direction": "Relatively more Headline+Product gaze than CTA",
    },
    {
        "balance": "B3_H_vs_P",
        "positive_direction": "Relatively more Headline gaze than Product",
        "negative_direction": "Relatively more Product gaze than Headline",
    },
])

display(interpretation)
interpretation.to_csv(
    OUTPUT_DIR / "ILR_balance_interpretation.csv",
    index=False
)


## 15. Output summary

In [ ]:

print("STEP 02 COMPLETE")
print("=" * 75)

print("Main analysis:")
print(" - Interpretable ILR balances")
print(" - Mixed-effects models")
print(" - Pairwise A-E contrasts")
print(" - BH/FDR correction")
print(" - Aitchison distances")
print(" - Image-level sensitivity")
print(" - Zero-replacement sensitivity")

print("\nGenerated files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)

print("\nMost important files to inspect:")
print(" 1. ILR_descriptive_statistics_by_condition.csv")
print(" 2. ILR_mixed_effects_coefficients.csv")
print(" 3. ILR_pairwise_mixed_model_contrasts_BH_FDR.csv")
print(" 4. condition_centroid_aitchison_distances.csv")
print(" 5. zero_replacement_sensitivity.csv")

print("\nMost important figures:")
print(" 1. ILR_distribution_B1_semantic_vs_other.png")
print(" 2. ILR_forest_C_vs_others_B1_semantic_vs_other.png")
print(" 3. aitchison_distance_heatmap.png")
print(" 4. image_trajectories_B1_semantic_vs_other.png")

print("\nNext folder:")
print("03_vlm_semantic_scoring")
